# Lending Risk Pipeline
Run the cells from top to bottom. This notebook calls reusable functions in `src/transformations.py`.

**Input:** `data/input/lending_risk.csv`  
**Output:** `data/output/`

## 1. Start Spark and set paths

In [16]:

from pathlib import Path
import sys


# Find the project folder
PROJECT = Path.cwd()

# If Jupyter opened in the notebooks folder, go up one level
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent

# Add the project folder to Python's import path
sys.path.insert(0, str(PROJECT))

print("Project folder:", PROJECT)
print("src folder exists:", (PROJECT / "src").exists())


Project folder: C:\lending_risk
src folder exists: True


In [17]:

from pathlib import Path
import sys

PROJECT = Path(r"C:\lending_risk")

sys.path.insert(0, str(PROJECT))

print("src exists:", (PROJECT / "src").exists())


src exists: True


In [18]:
from pathlib import Path
from pyspark.sql import SparkSession
from src.transformations import transform_data, make_summary

PROJECT = Path.cwd()
INPUT_FILE = PROJECT / "data" / "input" / "lending_risk.csv"
OUTPUT_DIR = PROJECT / "data" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("LendingRiskPipeline")
    .master("local[*]")
    .getOrCreate()
)
print("Spark version:", spark.version)
print("Input file:", INPUT_FILE)
print("Input file exists:", INPUT_FILE.exists())

Spark version: 4.2.0
Input file: C:\lending_risk\notebooks\data\input\lending_risk.csv
Input file exists: False


## 2. Read and preserve the raw input

In [19]:

from pathlib import Path
import sys

# Main project folder
PROJECT = Path(r"C:\lending_risk")

# Add project folder so Python can find src/
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

# Input and output paths
INPUT_FILE = PROJECT / "data" / "input" / "lending_risk.csv"
OUTPUT_DIR = PROJECT / "data" / "output"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT)
print("CSV exists:", INPUT_FILE.exists())
print("CSV path:", INPUT_FILE)


Project folder: C:\lending_risk
CSV exists: True
CSV path: C:\lending_risk\data\input\lending_risk.csv


In [20]:
# Read every source column as text first, so invalid source values are preserved.
raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(str(INPUT_FILE))
)
print("Raw rows:", raw_df.count())
raw_df.show(truncate=False)

raw_file = OUTPUT_DIR  /"raw_csv_folder"/"raw.csv"

raw_df.toPandas().to_csv(raw_file, index=False)

print("Raw data saved to:", raw_file)

Raw rows: 6
+-----------+----------------+------+----------+-------------+---------------+--------------------+
|facility_id|customer_segment|metric|period_end|actual_value |threshold_value|source_updated_at   |
+-----------+----------------+------+----------+-------------+---------------+--------------------+
|F001       |SME             |PD    |2026-09-30|3.2          |3.0            |2026-10-01T05:00:00Z|
|F002       |Corporate       |PD    |2026-09-30|2.7          |3.0            |2026-10-01T05:05:00Z|
|F003       |SME             |PD    |2026-09-30|not_available|3.0            |2026-10-01T05:10:00Z|
|F004       |NULL            |PD    |2026-09-30|3.1          |3.0            |2026-10-01T05:15:00Z|
|F005       |Corporate       |PD    |2026-09-30|3.1          |3.0            |2026-10-01T05:20:00Z|
|F005       |Corporate       |PD    |2026-09-30|3.5          |3.0            |2026-10-01T06:20:00Z|
+-----------+----------------+------+----------+-------------+---------------+----------

## 3. Transform, validate and deduplicate

In [21]:
detail_df, rejected_df, duplicates_df = transform_data(raw_df)
# print(transform_data(duplicates_df))
# detail, rejected_output, duplicate_output

print("Valid detail rows:", detail_df.count())
print("Rejected rows:", rejected_df.count())
print("Superseded duplicate rows:", duplicates_df.count())

detail_df.show(truncate=False)
rejected_df.select("facility_id", "rejection_reason").show(truncate=False)
duplicates_df.show(truncate=False)

Valid detail rows: 3
Rejected rows: 2
Superseded duplicate rows: 1
+-----------+----------------+------+----------+------------+---------------+-------------------+---------+
|facility_id|customer_segment|metric|period_end|actual_value|threshold_value|source_updated_at  |is_breach|
+-----------+----------------+------+----------+------------+---------------+-------------------+---------+
|F001       |SME             |PD    |2026-09-30|3.2         |3.0            |2026-10-01 07:00:00|true     |
|F002       |Corporate       |PD    |2026-09-30|2.7         |3.0            |2026-10-01 07:05:00|false    |
|F005       |Corporate       |PD    |2026-09-30|3.5         |3.0            |2026-10-01 08:20:00|true     |
+-----------+----------------+------+----------+------------+---------------+-------------------+---------+

+-----------+-----------------------------------------------------------------------------------+
|facility_id|rejection_reason                                                 

## 4. Build the report summary

In [22]:
summary_df = make_summary(detail_df)
summary_df.show(truncate=False)

+----------+----------------+------+--------------+-----------------------+-----------+--------------------+
|period_end|customer_segment|metric|facility_count|breached_facility_count|breach_rate|average_actual_value|
+----------+----------------+------+--------------+-----------------------+-----------+--------------------+
|2026-09-30|Corporate       |PD    |2             |1                      |0.5        |3.1                 |
|2026-09-30|SME             |PD    |1             |1                      |1.0        |3.2                 |
+----------+----------------+------+--------------+-----------------------+-----------+--------------------+



## 5. Export CSV files for Power BI
Spark writes a folder containing CSV part files for each output. The folder names are listed below.

In [23]:
def save_spark_df_to_csv(df, file_path):
    """Converts a Spark DataFrame to a Pandas DataFrame and saves it as a single CSV file."""
    # Convert to Pandas DataFrame
    pandas_df = df.toPandas()
    
    # Save to CSV without the row index, ensuring UTF-8 encoding
    pandas_df.to_csv(file_path, index=False, encoding="utf-8")

# Save all dataframes using the optimized Pandas approach
save_spark_df_to_csv(raw_df, OUTPUT_DIR / "raw.csv")
save_spark_df_to_csv(detail_df, OUTPUT_DIR / "detail.csv")
save_spark_df_to_csv(rejected_df, OUTPUT_DIR / "rejected.csv")
save_spark_df_to_csv(duplicates_df, OUTPUT_DIR / "duplicates.csv")
save_spark_df_to_csv(summary_df, OUTPUT_DIR / "summary.csv")

print("All CSV files saved to:", OUTPUT_DIR)


All CSV files saved to: C:\lending_risk\data\output


In [14]:
# detail_df.write.mode("overwrite").option("header", True).csv(str(OUTPUT_DIR / "detail"))
# rejected_df.coalesce(1).write.mode("overwrite").option("header", True).csv(str(OUTPUT_DIR / "rejected"))
# duplicates_df.coalesce(1).write.mode("overwrite").option("header", True).csv(str(OUTPUT_DIR / "duplicates"))
# summary_df.coalesce(1).write.mode("overwrite").option("header", True).csv(str(OUTPUT_DIR / "summary"))

# print("Outputs written to:", OUTPUT_DIR)
# print("For Power BI, open each folder and use its part-*.csv file.")

## 6. UnitTest Validations
Run pytest in a terminal from the project folder: `python -m pytest -q`.